<p style="text-align: right; margin: 0; line-height: 0;"><img width="100px" height="auto" src="../../.admin/img/eth_logo.png"></p>

<h1 style="text-align: center;">01 Data and Signals</h1>
<hr>


## **2. Representing Signals in Time and Frequency Domain**


<div class="alert alert-light">

  **Name:** (add your name here)
  
  **Matriculation Number:** (add your number here)
  
</div>


### 2.1 Introduction

A signal can be examined from two complementary perspectives: the **time domain** and the **frequency domain**. The time domain describes how a signal changes over time, whereas the frequency domain reveals the sinusoidal components that constitute the signal. A frequency-domain representation is commonly called a **spectrum**.

Both perspectives are essential in mobile computing. Wireless transmissions, audio recordings, and sensor measurements are acquired as time-varying signals, but their frequency content often exposes characteristics that are difficult to identify directly in the waveform. Examples include carrier frequencies in communication systems, pitches in audio, and periodic motion in sensor data.

The **Fourier transform** provides the mathematical connection between these domains by decomposing a signal into sinusoids with different frequencies, amplitudes, and phases. The inverse Fourier transform combines these components to reconstruct the time-domain signal.



The following visualization combines these ideas in one coordinate system: the front wall contains the time-domain waveform, the side wall contains the frequency-domain amplitudes, and the sinusoidal components appear inside the cube at their corresponding frequencies. 

In [ ]:
from base64 import b64encode
from pathlib import Path
from IPython.display import HTML, display

cube_locations = [
    Path("../../.admin/img/time_frequency_cube.html"),
    Path(".admin/img/time_frequency_cube.html"),
]
cube_path = next((path for path in cube_locations if path.exists()), None)
if cube_path is None:
    raise FileNotFoundError("time_frequency_cube.html was not found")

cube_data = b64encode(cube_path.read_bytes()).decode("ascii")
display(HTML(f"""
<div style="text-align: center;">
  <iframe src="data:text/html;base64,{cube_data}" width="520" height="390"
          style="border: 0; max-width: 100%;" sandbox="allow-scripts"
          title="Interactive 3D visualization of the time and frequency domains"></iframe>
  <p><small>Interactive time–frequency decomposition. Drag the cube to rotate it.</small></p>
</div>
"""))


The plots below show how three sinusoids combine to form a signal. Use the sliders to change the frequency, amplitude, or phase of one component at a time, and observe how the combined waveform changes on the left. The plot on the right shows the frequency and amplitude of each source component; phase is not displayed.

In [ ]:
from pathlib import Path
from runpy import run_path

_helper = next(root / ".admin/notebooks/time_frequency.py"
               for root in (Path.cwd(), *Path.cwd().parents)
               if (root / ".admin/notebooks/time_frequency.py").is_file())
_time_frequency = run_path(str(_helper))
controls = _time_frequency["show_time_frequency"]()


### 2.2 Fourier Transform: From Time to Frequency

#### 2.2.1 Continuous Fourier Transform

The previous example starts with three sinusoids and adds them to produce a waveform. The Fourier transform works in the opposite direction: it starts with a waveform and determines its frequency components. Each component needs both a strength and a phase. The strength describes its contribution to the signal, while the phase describes its position within a cycle. Changing either can change the combined waveform.

For a continuous-time signal $x(t)$, such as a microphone voltage, we use the **continuous Fourier transform**. Time $t$ and frequency $f$ are continuous variables, with $t$ measured in seconds and $f$ in hertz:

$$
X(f)=\int_{-\infty}^{\infty}x(t)e^{-j2\pi ft}\,dt.
$$

Here, $j$ is the imaginary unit, with $j^2=-1$. By Euler's formula,

$$
e^{-j2\pi ft}=\cos(2\pi ft)-j\sin(2\pi ft).
$$

The complex exponential is therefore a compact way to use a cosine and a sine together. For each frequency $f$, the transform multiplies the signal by these reference waves and adds the contributions over time. The integral is the continuous counterpart of a sum. The resulting complex value $X(f)$ contains both magnitude and phase information about that frequency.


#### 2.2.2 Why Can the Frequency Components Reproduce the Original Signal?

One sine wave has a very restricted shape: the same smooth oscillation repeats indefinitely. It cannot by itself describe a short pulse, a sharp edge, or an irregular waveform. The flexibility comes from adding many sinusoids with different frequencies, strengths, and phases. At one time, their values can reinforce each other; at another, they can cancel. The sum does not have to look like any of its individual components.

To see why this can be enough to describe a whole signal, first imagine drawing $x(t)$ using a row of narrow, adjacent rectangles. Give each rectangle the height of the signal in that small time interval. The result is a staircase approximation. Making the rectangles narrower allows the staircase to follow finer details of the waveform.

Now consider just one of those rectangles. As notebook 01.3 will demonstrate, a rectangular pulse can itself be approximated by combining sinusoids. Their contributions reinforce within the pulse and largely cancel outside it. Including higher frequencies allows the transitions to become sharper. Exact vertical edges require frequencies extending without limit; a finite bandwidth leaves rounded transitions and ringing.

This gives us a way to connect the two representations: **a waveform can be approximated by a row of pulses, and those pulses can be approximated by sinusoids.** Adding all the sinusoidal contributions therefore gives an approximation to the original waveform. Contributions at the same frequency can be combined into a single sinusoid with an appropriate strength and phase. We do not need to store a separate spectrum for every pulse.

The Fourier transform determines these combined contributions directly from $x(t)$, without first constructing the rectangles. The **inverse Fourier transform** adds them back together:

$$
x(t)=\int_{-\infty}^{\infty}X(f)e^{j2\pi ft}\,df.
$$

The integral includes contributions from a continuous range of frequencies. For a periodic signal, the corresponding representation is a **Fourier series**, which uses a sum of harmonics at integer multiples of the fundamental frequency.

The phases $\phi_m$ control where the components reinforce and where they cancel. Shifting a pulse in time changes these phases even though its magnitude spectrum remains the same. Keeping only the magnitudes would lose the information needed to place the pulse correctly. The phase sliders above demonstrate the same principle: they change the waveform without changing the displayed component amplitudes.

For finite-duration, piecewise smooth signals, the full Fourier representation reconstructs $x(t)$ wherever it is continuous. At a jump, it gives the average of the values on either side. The practical distinction is therefore between retaining the full frequency information and keeping only a limited bandwidth. Notebook [01.3, Fourier Approximation of a Rectangular Pulse](01.3_Fourier_Approximation_of_a_Rectangular_Pulse.ipynb), explores this distinction in detail.


### 2.3 Examples


A periodic target can be approximated by adding harmonic signals. In all three graphs below, harmonic $m$ has the equation:

$$s_m(t)=A_m\sin(2\pi f_m t+\phi_m),$$

and the approximation shown in red is

$$\hat{x}(t)=c_0+\sum_{m=1}^{7}s_m(t),$$

Thus, every row of controls describes one complete harmonic equation: its **amplitude** $A_m$, **frequency** $f_m$ in Hz, and **phase** $\phi_m$ in degrees. The fixed term $c_0$ is the target's average value. The presets use $f_m=m f_0$ with fundamental frequency $f_0=1$ Hz and coefficients obtained from the target's Fourier series. Seven harmonics are used: this is enough to make the characteristic shapes recognizable while keeping all parameters manageable.



In [ ]:
from pathlib import Path
from runpy import run_path

_helper = next(root / ".admin/notebooks/harmonic_examples.py"
               for root in (Path.cwd(), *Path.cwd().parents)
               if (root / ".admin/notebooks/harmonic_examples.py").is_file())
run_path(str(_helper))["show_harmonic_examples"]()


---

### 2.4 Hands-On Exercise

<div class="alert alert-block alert-info">Fill in your responses and save this notebook. See the <a href="../../03_Assignments/Submission-Guidelines.md">submission guidelines</a> for how to submit your work.</div>


**For each example in Section 2.3, compare the dashed black target with the red sum. The seven equations shown with each example give the reference values, but one slider has a deliberately incorrect preset. Compare the sliders with the equations, find each mismatch, and correct it.**

Answer: 
(Submit with the corrected images from Section 2.3.)



**After correcting the errors in Section 2.3, do the seven-harmonic approximations match their targets exactly?**

Answer:
(answer here)



**What is the highest harmonic frequency used in the approximations in Section 2.3?**

Answer:
(answer here)



**What frequency range would be required to reproduce the target waveforms in Section 2.3 exactly?**

Answer:
(answer here)




**OPTIONAL: Did you read through and understand all the tasks in this Notebook? Are there any shortcomings, missing parts, or sections that should be improved?**

 (please help the team by telling us what part was not understood, and we will work on it)